In [1]:
# 데이터 준비

!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 80.2M    0 98304    0     0  62713      0  0:22:21  0:00:01  0:22:20 63055
  1 80.2M    1 1488k    0     0   579k      0  0:02:21  0:00:02  0:02:19  581k
  5 80.2M    5 4480k    0     0  1204k      0  0:01:08  0:00:03  0:01:05 1207k
  8 80.2M    8 7072k    0     0  1531k      0  0:00:53  0:00:04  0:00:49 1533k
 11 80.2M   11 9616k    0     0  1724k      0  0:00:47  0:00:05  0:00:42 1927k
 15 80.2M   15 12.2M    0     0  1861k      0  0:00:44  0:00:06  0:00:38 2409k
 17 80.2M   17 14.4M    0     0  1935k      0  0:00:42  0:00:07  0:00:35 2621k
 20 80.2M   20 16.5M    0     0  1971k      0  0:00:41  0:00:08  0:00:33 2557k
 23 80.2M   23 18.5M    0     0  1971k      0  0:00

In [ ]:
# 다운로드한 tar.gz 파일 압축 해제 ( tar.gz : 리눅스 계열에서 사용하는 압축파일 )

import tarfile

with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall('data-files')

In [7]:
# 불필요한 폴더 제거 (aclimdb/train/unsup 폴더 제거)
import shutil

shutil.rmtree('data-files/aclimdb/train/unsup')

In [8]:
# validation 데이터셋 준비 ( train 폴더의 파일 20%를 val 폴더로 이동 )
import os, pathlib, shutil, random

base_dir = pathlib.Path("data-files/aclimdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    random.Random(42).shuffle(files)
    num_val_samples = int(0.2 * len(files))
    val_files = files[-num_val_samples:]
    for fname in val_files:
        shutil.move(train_dir / category / fname,
                    val_dir / category / fname)

In [ ]:
#####

In [1]:
from tensorflow import keras as tf_keras

train_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/train', batch_size=32
)
validation_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/val', batch_size=32
)
test_dataset = tf_keras.utils.text_dataset_from_directory(
    'data-files/aclimdb/test', batch_size=32
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [ ]:
# 데이터셋 동작 확인
for inputs, labels in train_dataset:
    print( inputs[0] )
    print( labels[0] )
    break

tf.Tensor(b'Modern viewers know this little film primarily as the model for the remake, "The Money Pit." Older viewers today watch it with wisps of nostalgia: Cary Grant, Myrna Loy, and Melvyn Douglas were all "superstars" in an easier, less complicated era. Or was it? Time, of course, has a way of modifying perspectives, and with so many films today verily ulcerating with social and political commentary, there is a natural curiosity to wonder about controversy in older, seemingly less provocative films. In "Mr. Blandings Builds His Dream House," there may, therefore, be more than what audiences were looking for in 1948. There is political commentary, however subtle. Finding a house in the late 40s was a truly exasperating experience, only lightly softened by the coming of Levittowns and the like. Politics in the movie? The Blandings children always seem to be talking about progressive ideas being taught to them in school (which in real life would get teachers accused of communism). In

In [2]:
# BoW 모델 기반 텍스트 데이터 인코딩 도구 학습

text_vectorization = tf_keras.layers.TextVectorization(
    max_tokens=20000, # 단어 사전에 포함될 단어 갯수 (빈도수 높은 순)

    # output_mode='int' # 각 단어의 단어 사전에 지정된 번호 인코딩
    output_mode="multi_hot" # 단어가 있는 곳에 갯수와 관계 없이 1로 인코딩
    # output_mode='count' # 단어가 있는 곳에 갯수를 인코딩
    # output_mode='tf-idf' # 단어가 있는 곳에 tf-idf 빈도 값 인코딩
)

only_text_dataset = train_dataset.map(lambda x, y: x)
text_vectorization.adapt( only_text_dataset ) # 학습을 통해 단어 사전 구성

In [3]:
# 데이터 셋의 각 데이터에 대해 인코딩 처리

encoded_unigram_train_dataset = train_dataset.map( lambda x, y: (text_vectorization(x), y) )
encoded_unigram_validation_dataset = validation_dataset.map( lambda x, y: (text_vectorization(x), y) )
encoded_unigram_test_dataset = test_dataset.map( lambda x, y: (text_vectorization(x), y) )

In [19]:
# 인코딩된 데이터셋 확인

for inputs, labels in encoded_unigram_train_dataset:
    print( inputs[-1][50:70] )
    break

tf.Tensor([0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0], shape=(20,), dtype=int64)


In [4]:
# 모델 구조 설계
inputs = tf_keras.layers.Input(shape=(20000, ))
x = tf_keras.layers.Dense(32, activation='relu')(inputs)
x = tf_keras.layers.Dropout(0.5)(x)
outputs = tf_keras.layers.Dense(1, activation='sigmoid')(x)

model = tf_keras.Model(inputs, outputs)

# 모델 학습 설계
model.compile(loss="binary_crossentropy",
              optimizer=tf_keras.optimizers.Adam(),
              metrics=['accuracy'])

# 모델 저장 콜백 만들기
callbacks = [
    tf_keras.callbacks.ModelCheckpoint('models/imdb-dense-model.keras', save_best_only=True)
]

# 모델 훈련
history = model.fit(encoded_unigram_train_dataset, epochs=10, 
                    validation_data=encoded_unigram_validation_dataset, 
                    callbacks=callbacks)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 42ms/step - accuracy: 0.8439 - loss: 0.3672 - val_accuracy: 0.8930 - val_loss: 0.2582
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 15s 25ms/step - accuracy: 0.9293 - loss: 0.1910 - val_accuracy: 0.8950 - val_loss: 0.2581
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.9552 - loss: 0.1285 - val_accuracy: 0.8854 - val_loss: 0.2971
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9712 - loss: 0.0863 - val_accuracy: 0.8880 - val_loss: 0.3277
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9792 - loss: 0.0633 - val_accuracy: 0.8916 - val_loss: 0.3567
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9845 - loss: 0.0481 - val_accuracy: 0.8924 - val_loss: 0.3864
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9875 - loss: 0.0378 - val_accuracy: 0.8912 - val_loss: 0.4273
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - accuracy: 0.9897 - loss: 0.0297 - 

In [6]:
best_model = tf_keras.models.load_model('models/imdb-dense-model.keras')
best_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       640,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,920,197 (7.32 MB)

 Trainable params: 640,065 (2.44 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,280,132 (4.88 MB)

In [7]:
best_model.evaluate(encoded_unigram_test_dataset)

782/782 ━━━━━━━━━━━━━━━━━━━━ 70s 88ms/step - accuracy: 0.8834 - loss: 0.2900


[0.28999194502830505, 0.8833600282669067]